# SEC Company Facts: покрытие, сопоставление и качество данных

Воспроизводимый анализ локального архива SEC Company Facts и компаний из `train.csv`.
Исходные файлы читаются без изменений. Все результаты ниже получены при последовательной обработке JSON-файлов; интернет и внешние API не используются.

## Context & Methods

### Ключевые допущения

- Единица списка интереса — уникальный идентификатор компании в `train.csv`, а не строка ценового временного ряда.
- Приоритет сопоставления: CIK → локальный точный ticker↔CIK → точное нормализованное название. В этом наборе локальная связь ticker↔CIK извлекается только из SEC-концепта `dei:EntityTradingSymbol` внутри Company Facts.
- Один тикер подтверждается только тогда, когда он ведёт ровно к одному CIK. Исторические или неоднозначные символы вынесены отдельно.
- Полная проверка читаемости и верхнеуровневой структуры выполняется для всех JSON. Детальная fact-level диагностика выполняется для подтверждённо сопоставленных компаний — это явно отделено от полного каталога.
- Отрицательные значения считаются наблюдением, а не автоматической ошибкой.

### 1. Конфигурация и импорты

Пути сначала проверяются как указанные абсолютные `/data/...`, затем относительно корня проекта. Все параметры собраны в одной ячейке.

In [1]:
from pathlib import Path
from collections import Counter
from IPython.display import display, Markdown
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 90)
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()

def resolve_input(absolute_candidate, relative_candidate):
    absolute_candidate = Path(absolute_candidate)
    relative_candidate = PROJECT_ROOT / relative_candidate
    if absolute_candidate.exists():
        return absolute_candidate.resolve()
    if relative_candidate.exists():
        return relative_candidate.resolve()
    raise FileNotFoundError(f"Не найдено ни {absolute_candidate}, ни {relative_candidate}")

COMPANYFACTS_DIR = resolve_input("/data/company_raw/companyfacts", "data/company_raw/companyfacts")
TRAIN_CSV = resolve_input("/data/train.csv", "data/train.csv")
OUTPUT_NOTEBOOK = PROJECT_ROOT / "company_analisys.ipynb"

print("Корень проекта:", PROJECT_ROOT)
print("Company Facts:", COMPANYFACTS_DIR)
print("Train CSV:", TRAIN_CSV)

Корень проекта: /Users/konstantinmelnikov/Desktop/work/portfolio optimization
Company Facts: /Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/companyfacts
Train CSV: /Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/train.csv


## Data

### 2. Инвентаризация JSON и схема `train.csv`

Сначала исследуем файлы и CSV без предположений о названиях колонок.

In [2]:
json_files = sorted(COMPANYFACTS_DIR.rglob("*.json"))
total_json_bytes = sum(path.stat().st_size for path in json_files)
inventory_summary = pd.DataFrame({
    "metric": ["JSON files", "Total bytes", "Total GiB"],
    "value": [len(json_files), total_json_bytes, total_json_bytes / 1024**3],
})
display(inventory_summary.style.format({"value": lambda x: f"{x:,.2f}" if isinstance(x, float) else f"{x:,}"}))

train_raw = pd.read_csv(TRAIN_CSV)
display(pd.DataFrame({
    "column": train_raw.columns,
    "dtype": train_raw.dtypes.astype(str).values,
    "non_null": train_raw.notna().sum().values,
    "null_rate_pct": (train_raw.isna().mean().values * 100),
}).style.format({"null_rate_pct": "{:.3f}"}))
display(train_raw.head(8))
print(f"Строк в train.csv: {len(train_raw):,}; колонок: {train_raw.shape[1]}")

,metric,value
0,JSON files,"20,290.00"
1,Total bytes,"19,284,675,755.00"
2,Total GiB,17.96


,column,dtype,non_null,null_rate_pct
0,Ticker,object,213850,0.000
1,Date,object,213850,0.000
2,Adj Close,float64,213850,0.000


,Ticker,Date,Adj Close
0,A,2000-01-03,43.04
1,A,2000-01-04,39.75
2,A,2000-01-05,37.28
3,A,2000-01-06,35.86
4,A,2000-01-07,38.85
5,A,2000-01-10,41.21
6,A,2000-01-11,40.65
7,A,2000-01-12,39.82


Строк в train.csv: 213,850; колонок: 3


### 3. Автоматическое исследование структуры SEC Company Facts

Типичный файл содержит:

- `cik` — Central Index Key эмитента;
- `entityName` — имя юридического лица;
- `facts` — словарь taxonomy (`us-gaap`, `dei`, `ifrs-full` и др.);
- внутри taxonomy ключ — имя XBRL-концепта, а `label` и `description` объясняют показатель;
- `units` группирует наблюдения по единице (`USD`, `shares`, `USD/shares`, `pure` и др.);
- отдельный факт может содержать период `start`–`end`, значение `val`, accession `accn`, финансовый год `fy`, период `fp`, форму `form`, дату публикации `filed` и стандартный SEC frame `frame`.

Ниже структура определяется по нескольким реально прочитанным файлам, без вывода больших сырых объектов.

In [3]:
def compact_structure(path):
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    taxonomies = list((data.get("facts") or {}).keys())
    sample_taxonomy = taxonomies[0] if taxonomies else None
    sample_concept = None
    concept_keys = []
    units = []
    fact_fields = []
    if sample_taxonomy:
        concepts = data["facts"].get(sample_taxonomy) or {}
        sample_concept = next(iter(concepts), None)
        if sample_concept:
            concept = concepts[sample_concept] or {}
            concept_keys = list(concept.keys())
            units = list((concept.get("units") or {}).keys())
            if units and concept["units"].get(units[0]):
                fact_fields = list(concept["units"][units[0]][0].keys())
    return {
        "file": path.name,
        "top_level_keys": list(data.keys()),
        "cik": data.get("cik"),
        "entityName": data.get("entityName"),
        "taxonomies": taxonomies,
        "sample_concept": f"{sample_taxonomy}:{sample_concept}",
        "concept_fields": concept_keys,
        "sample_units": units,
        "fact_fields": fact_fields,
    }

structure_examples = pd.DataFrame([compact_structure(path) for path in json_files[:3]])
display(structure_examples)

,file,top_level_keys,cik,entityName,taxonomies,sample_concept,concept_fields,sample_units,fact_fields
0,CIK0000001750.json,"[cik, entityName, facts]",1750,AAR CORP.,"[dei, us-gaap, ffd, ecd]",dei:EntityCommonStockSharesOutstanding,"[label, description, units]",[shares],"[end, val, accn, fy, fp, form, filed, frame]"
1,CIK0000001800.json,"[cik, entityName, facts]",1800,ABBOTT LABORATORIES,"[dei, us-gaap, ffd]",dei:EntityCommonStockSharesOutstanding,"[label, description, units]",[shares],"[end, val, accn, fy, fp, form, filed, frame]"
2,CIK0000001961.json,"[cik, entityName, facts]",1961,GEMAXEL INC. (formerly known as Worlds Inc.),"[dei, invest, us-gaap]",dei:EntityCommonStockSharesOutstanding,"[label, description, units]",[shares],"[end, val, accn, fy, fp, form, filed, frame]"


### 4. Последовательное построение каталога и локального ticker↔CIK

Каждый JSON загружается, обрабатывается и освобождается отдельно. Ошибка одного файла не останавливает проход. Помимо каталога компаний извлекается `EntityTradingSymbol`; только файлы с тикерами из `train.csv` разворачиваются до уровня отдельных фактов для последующего анализа.

In [4]:
def normalize_cik(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return pd.NA
    text = str(value).strip()
    if re.fullmatch(r"\d+(?:\.0+)?", text):
        text = text.split(".")[0]
    digits = re.sub(r"\D", "", text)
    return digits.zfill(10) if digits else pd.NA

def normalize_ticker(value):
    if value is None:
        return None
    ticker = str(value).strip().upper()
    return ticker if re.fullmatch(r"[A-Z][A-Z0-9.\-]{0,14}", ticker) else None

def find_column(columns, candidates):
    normalized = {re.sub(r"[^a-z0-9]", "", str(c).lower()): c for c in columns}
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", candidate.lower())
        if key in normalized:
            return normalized[key]
    return None

def extract_trading_symbols(data):
    symbols = set()
    for taxonomy, concepts in (data.get("facts") or {}).items():
        for concept_name, concept_data in (concepts or {}).items():
            if "tradingsymbol" not in concept_name.lower():
                continue
            for observations in (concept_data.get("units") or {}).values():
                for observation in observations or []:
                    symbol = normalize_ticker(observation.get("val"))
                    if symbol:
                        symbols.add(symbol)
    return sorted(symbols)

def flatten_facts(data, cik, entity_name, json_path):
    rows = []
    for taxonomy, concepts in (data.get("facts") or {}).items():
        for concept_name, concept_data in (concepts or {}).items():
            label = concept_data.get("label")
            description = concept_data.get("description")
            for unit, observations in (concept_data.get("units") or {}).items():
                for observation in observations or []:
                    rows.append({
                        "cik": cik,
                        "entity_name": entity_name,
                        "taxonomy": taxonomy,
                        "concept": concept_name,
                        "label": label,
                        "description": description,
                        "unit": unit,
                        "start": observation.get("start"),
                        "end": observation.get("end"),
                        "val": observation.get("val"),
                        "accn": observation.get("accn"),
                        "fy": observation.get("fy"),
                        "fp": observation.get("fp"),
                        "form": observation.get("form"),
                        "filed": observation.get("filed"),
                        "frame": observation.get("frame"),
                        "json_path": str(json_path),
                    })
    return rows

ticker_column = find_column(train_raw.columns, ["ticker", "symbol", "trading_symbol"])
cik_column = find_column(train_raw.columns, ["cik", "sec_cik"])
name_column = find_column(train_raw.columns, ["company_name", "company", "name", "entity_name"])

if ticker_column:
    train_identifiers = sorted({t for t in train_raw[ticker_column].map(normalize_ticker) if t})
elif cik_column:
    train_identifiers = sorted(train_raw[cik_column].map(normalize_cik).dropna().unique())
else:
    raise ValueError("В train.csv нет распознанного CIK, ticker или названия компании.")

train_ticker_set = set(train_identifiers) if ticker_column else set()
FACT_COLUMNS = ["cik", "entity_name", "taxonomy", "concept", "label", "description", "unit", "start", "end", "val", "accn", "fy", "fp", "form", "filed", "frame", "json_path"]
CACHE_DIR = PROJECT_ROOT / ".cache_company_analysis"
CACHE_PATH = CACHE_DIR / "companyfacts_scan.pkl"
source_signature = {
    "file_count": len(json_files),
    "total_bytes": total_json_bytes,
    "latest_mtime_ns": max((path.stat().st_mtime_ns for path in json_files), default=0),
}

cache_payload = None
if CACHE_PATH.exists():
    try:
        candidate = pd.read_pickle(CACHE_PATH)
        if candidate.get("source_signature") == source_signature:
            cache_payload = candidate
            print("Используется валидный локальный cache производного каталога:", CACHE_PATH)
    except Exception as exc:
        print("Cache проигнорирован:", type(exc).__name__, str(exc)[:200])

if cache_payload is None:
    catalog_rows, ticker_map_rows, error_rows, matched_fact_rows, fallback_fact_rows = [], [], [], [], []
    fallback_company_count = 0
    for path in tqdm(json_files, desc="Чтение Company Facts", unit="file"):
        try:
            with path.open("r", encoding="utf-8") as handle:
                data = json.load(handle)
            cik = normalize_cik(data.get("cik"))
            entity_name = data.get("entityName")
            taxonomies = data.get("facts") or {}
            symbols = extract_trading_symbols(data)
            concept_count = sum(len(concepts or {}) for concepts in taxonomies.values())
            # Полное число фактов считается только для выбранных компаний после flatten;
            # для общего каталога это не требуется и заметно удлиняет проход по 19+ ГБ.
            fact_count = pd.NA
            catalog_rows.append({
                "cik": cik, "entityName": entity_name, "json_path": str(path),
                "file_size_bytes": path.stat().st_size, "taxonomies": tuple(sorted(taxonomies.keys())),
                "concept_count": concept_count, "fact_count": fact_count, "trading_symbols": tuple(symbols),
            })
            for symbol in symbols:
                ticker_map_rows.append({"ticker": symbol, "cik": cik, "sec_entity_name": entity_name, "json_path": str(path)})
            if train_ticker_set.intersection(symbols):
                matched_fact_rows.extend(flatten_facts(data, cik, entity_name, path))
            if fallback_company_count < 2 and not pd.isna(cik) and taxonomies:
                fallback_fact_rows.extend(flatten_facts(data, cik, entity_name, path))
                fallback_company_count += 1
        except Exception as exc:
            error_rows.append({"json_path": str(path), "error_type": type(exc).__name__, "error": str(exc)[:500]})

    cache_payload = {
        "source_signature": source_signature,
        "catalog": pd.DataFrame(catalog_rows),
        "ticker_map": pd.DataFrame(ticker_map_rows, columns=["ticker", "cik", "sec_entity_name", "json_path"]),
        "read_errors": pd.DataFrame(error_rows, columns=["json_path", "error_type", "error"]),
        "facts": pd.DataFrame(matched_fact_rows, columns=FACT_COLUMNS),
        "fallback_facts": pd.DataFrame(fallback_fact_rows, columns=FACT_COLUMNS),
    }
    CACHE_DIR.mkdir(exist_ok=True)
    pd.to_pickle(cache_payload, CACHE_PATH)
    print("Создан локальный cache производных результатов:", CACHE_PATH)

catalog = cache_payload["catalog"]
ticker_map = cache_payload["ticker_map"]
read_errors = cache_payload["read_errors"]
facts = cache_payload["facts"]
fallback_facts = cache_payload["fallback_facts"]

print(f"Успешно прочитано: {len(catalog):,} из {len(json_files):,}; ошибок: {len(read_errors):,}")
print(f"Локальных пар ticker–CIK: {len(ticker_map):,}; развёрнутых фактов кандидатов: {len(facts):,}")
display(read_errors.head(10))

Используется валидный локальный cache производного каталога: /Users/konstantinmelnikov/Desktop/work/portfolio optimization/.cache_company_analysis/companyfacts_scan.pkl
Успешно прочитано: 20,290 из 20,290; ошибок: 0
Локальных пар ticker–CIK: 0; развёрнутых фактов кандидатов: 0


,json_path,error_type,error


## Results

### 5. Сопоставление `train.csv` с Company Facts

Повторение тикера в `train.csv` по разным датам ожидаемо и не считается дубликатом компании. Подтверждение по `EntityTradingSymbol` допускается только для однозначной пары ticker–CIK. Fuzzy matching не применяется, если в `train.csv` отсутствует название компании.

In [5]:
company_list = pd.DataFrame({"train_identifier": train_identifiers})
if ticker_column:
    company_list["train_company_name"] = pd.NA
    ticker_candidates = (
        ticker_map.dropna(subset=["ticker", "cik"])
        .drop_duplicates(["ticker", "cik", "json_path"])
    )
    candidate_counts = ticker_candidates.groupby("ticker")["cik"].nunique()
    unambiguous_tickers = set(candidate_counts[candidate_counts == 1].index)
    confirmed = (
        company_list[company_list["train_identifier"].isin(unambiguous_tickers)]
        .merge(ticker_candidates, left_on="train_identifier", right_on="ticker", how="left")
        .drop(columns="ticker")
        .drop_duplicates(["train_identifier", "cik"])
    )
    confirmed["match_method"] = "exact SEC dei:EntityTradingSymbol"
    ambiguous_matches = ticker_candidates[ticker_candidates["ticker"].isin(candidate_counts[candidate_counts > 1].index)].copy()
    unmatched = company_list[~company_list["train_identifier"].isin(set(confirmed["train_identifier"]))].copy()
else:
    # Универсальная ветка для CIK-содержащего train.csv.
    company_list["train_identifier"] = train_raw[cik_column].map(normalize_cik).dropna().unique()
    company_list["train_company_name"] = train_raw[name_column] if name_column else pd.NA
    confirmed = company_list.merge(catalog, left_on="train_identifier", right_on="cik", how="inner")
    confirmed = confirmed.rename(columns={"entityName": "sec_entity_name"})
    confirmed["match_method"] = "exact CIK"
    ambiguous_matches = pd.DataFrame(columns=["ticker", "cik", "sec_entity_name", "json_path"])
    unmatched = company_list[~company_list["train_identifier"].isin(set(confirmed["train_identifier"]))].copy()

confirmed = confirmed[["train_identifier", "train_company_name", "cik", "sec_entity_name", "match_method", "json_path"]]
confirmed_ciks = set(confirmed["cik"].dropna())
unused_companyfacts = catalog[~catalog["cik"].isin(confirmed_ciks)].copy()
fuzzy_matches = pd.DataFrame(columns=["train_identifier", "candidate_cik", "candidate_name", "similarity"])

train_company_count = len(company_list)
json_company_count = catalog["cik"].nunique(dropna=True)
match_count = len(confirmed)
match_rate = match_count / train_company_count if train_company_count else np.nan

match_summary = pd.DataFrame({
    "metric": ["Уникальных компаний/идентификаторов в train", "Уникальных CIK в JSON", "Подтверждённых совпадений", "Доля совпадений", "Несовпавших", "Неоднозначных ticker–CIK"],
    "value": [f"{train_company_count:,}", f"{json_company_count:,}", f"{match_count:,}", f"{match_rate:.1%}", f"{len(unmatched):,}", f"{ambiguous_matches['ticker'].nunique() if not ambiguous_matches.empty else 0:,}"],
})
display(match_summary)
if ticker_column and ticker_map.empty:
    display(Markdown(
        "**Ограничение:** `train.csv` содержит только тикеры, а в папке `data` и в SEC Company Facts "
        "нет локального ticker↔CIK. Подтверждённое сопоставление без внешнего или предоставленного "
        "справочника невозможно. Нужна таблица минимум с колонками `Ticker` и `CIK`."
    ))

print("Подтверждённые совпадения:")
display(confirmed.head(60))
print("Компании train без подтверждённого совпадения:")
display(unmatched.head(60))
print("Неоднозначные соответствия ticker–CIK:")
display(ambiguous_matches.head(30))
print("Company Facts, не используемые train (компактный preview):")
display(unused_companyfacts[["cik", "entityName", "json_path"]].head(20))
print("Предположительные fuzzy-совпадения (не строятся без названий в train):")
display(fuzzy_matches)

duplicate_checks = pd.DataFrame({
    "check": ["Дубликаты строк train", "Дубликаты ключа Ticker+Date", "CIK с несколькими JSON-файлами", "Ticker, ведущие к нескольким CIK"],
    "count": [
        int(train_raw.duplicated().sum()),
        int(train_raw.duplicated([ticker_column, find_column(train_raw.columns, ["date"]) ]).sum()) if ticker_column and find_column(train_raw.columns, ["date"]) else np.nan,
        int((catalog.groupby("cik")["json_path"].nunique() > 1).sum()),
        int((ticker_map.groupby("ticker")["cik"].nunique() > 1).sum()) if not ticker_map.empty else 0,
    ],
})
display(duplicate_checks)

,metric,value
0,Уникальных компаний/идентификаторов в train,50
1,Уникальных CIK в JSON,"20,220"
2,Подтверждённых совпадений,0
3,Доля совпадений,0.0%
4,Несовпавших,50
5,Неоднозначных ticker–CIK,0


**Ограничение:** `train.csv` содержит только тикеры, а в папке `data` и в SEC Company Facts нет локального ticker↔CIK. Подтверждённое сопоставление без внешнего или предоставленного справочника невозможно. Нужна таблица минимум с колонками `Ticker` и `CIK`.

Подтверждённые совпадения:


,train_identifier,train_company_name,cik,sec_entity_name,match_method,json_path


Компании train без подтверждённого совпадения:


...

Company Facts, не используемые train (компактный preview):


,cik,entityName,json_path
0,0000001750,AAR CORP.,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
1,0000001800,ABBOTT LABORATORIES,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
2,0000001961,GEMAXEL INC. (formerly known as Worlds Inc.),/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
3,0000002034,ACETO CORP,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
4,0000002098,ACME UNITED CORP,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
5,0000002110,,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
6,0000002178,"ADAMS RESOURCES & ENERGY, INC.",/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
7,0000002186,BK Technologies Corp,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
8,0000002230,,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...
9,0000002488,ADVANCED MICRO DEVICES INC,/Users/konstantinmelnikov/Desktop/work/portfolio optimization/data/company_raw/company...


Предположительные fuzzy-совпадения (не строятся без названий в train):


,train_identifier,candidate_cik,candidate_name,similarity


,check,count
0,Дубликаты строк train,0
1,Дубликаты ключа Ticker+Date,0
2,CIK с несколькими JSON-файлами,0
3,"Ticker, ведущие к нескольким CIK",0


### 6. Две компании в читаемом виде

Компании выбираются автоматически среди подтверждённых совпадений. Если подтверждённых совпадений нет из-за отсутствия локального ticker↔CIK, используются первые две валидные JSON-компании как **иллюстративные, не сопоставленные с train**. Для понятных показателей используется набор распространённых XBRL-концептов; показываются только небольшие последние срезы, а не сырой JSON.

In [6]:
matched_facts = facts[facts["cik"].isin(confirmed_ciks)].copy()
for frame in [matched_facts, fallback_facts]:
    frame["end_date"] = pd.to_datetime(frame["end"], errors="coerce")
    frame["start_date"] = pd.to_datetime(frame["start"], errors="coerce")
    frame["filed_date"] = pd.to_datetime(frame["filed"], errors="coerce")

example_pool = matched_facts if not matched_facts.empty else fallback_facts
quality_facts = matched_facts if not matched_facts.empty else fallback_facts
quality_scope_label = "confirmed train companies" if not matched_facts.empty else "2 illustrative JSON companies (no confirmed train matches)"

preferred_concepts = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "Revenues"],
    "net income": ["NetIncomeLoss", "ProfitLoss"],
    "assets": ["Assets"],
    "liabilities": ["Liabilities"],
    "equity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "cash": ["CashAndCashEquivalentsAtCarryingValue", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"],
    "operating cash flow": ["NetCashProvidedByUsedInOperatingActivities"],
    "capital expenditure": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "EPS": ["EarningsPerShareDiluted", "EarningsPerShareBasic"],
    "shares outstanding": ["EntityCommonStockSharesOutstanding", "CommonStocksIncludingAdditionalPaidInCapitalMember"],
}

if not confirmed.empty:
    example_ids = confirmed.sort_values("train_identifier").head(2)[["train_identifier", "cik"]]
else:
    example_ids = pd.DataFrame({
        "train_identifier": "illustrative only — not matched",
        "cik": fallback_facts["cik"].dropna().drop_duplicates().head(2).tolist(),
    })
example_summary_rows, example_fact_parts = [], []
for row in example_ids.itertuples(index=False):
    subset = example_pool[example_pool["cik"] == row.cik].copy()
    example_summary_rows.append({
        "train_identifier": row.train_identifier,
        "cik": row.cik,
        "company": subset["entity_name"].dropna().iloc[0] if subset["entity_name"].notna().any() else pd.NA,
        "concept_count": subset[["taxonomy", "concept"]].drop_duplicates().shape[0],
        "taxonomies": ", ".join(sorted(subset["taxonomy"].dropna().unique())),
        "earliest_end": subset["end_date"].min(),
        "latest_end": subset["end_date"].max(),
    })
    for readable_name, candidates in preferred_concepts.items():
        available = [c for c in candidates if c in set(subset["concept"])]
        if not available:
            continue
        chosen = available[0]
        recent = subset[subset["concept"] == chosen].sort_values(["end_date", "filed_date"]).tail(2).copy()
        recent["metric"] = readable_name
        example_fact_parts.append(recent)

example_summary = pd.DataFrame(example_summary_rows)
example_facts = pd.concat(example_fact_parts, ignore_index=True) if example_fact_parts else pd.DataFrame()
display(example_summary)

if not example_facts.empty:
    readable_examples = example_facts.rename(columns={
        "entity_name": "company", "val": "value", "fy": "fiscal_year", "fp": "fiscal_period", "accn": "accession_number"
    })[["company", "cik", "metric", "concept", "label", "unit", "start", "end", "value", "form", "fiscal_year", "fiscal_period", "filed", "frame", "accession_number"]]
    for company in readable_examples["company"].drop_duplicates():
        display(Markdown(f"#### {company}"))
        display(readable_examples[readable_examples["company"] == company].reset_index(drop=True))
else:
    display(Markdown("Нет валидных фактов для компактных примеров."))

,train_identifier,cik,company,concept_count,taxonomies,earliest_end,latest_end
0,illustrative only — not matched,0000001750,AAR CORP.,497,"dei, ecd, ffd, us-gaap",2008-05-31,2026-07-16
1,illustrative only — not matched,0000001800,ABBOTT LABORATORIES,482,"dei, ffd, us-gaap",2006-12-31,2026-04-24


#### AAR CORP.

,company,cik,metric,concept,label,unit,start,end,value,form,fiscal_year,fiscal_period,filed,frame,accession_number
0,AAR CORP.,0000001750,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,"Revenue from Contract with Customer, Excluding Assessed Tax",USD,2019-12-01,2020-02-29,9.800000e+06,10-Q,2020.0,Q3,2020-03-25,CY2020Q1,0001104659-20-037857
1,AAR CORP.,0000001750,net income,NetIncomeLoss,Net Income (Loss) Attributable to Parent,USD,2025-06-01,2026-05-31,1.877000e+08,10-K,2026.0,FY,2026-07-22,None,0001104659-26-085459
2,AAR CORP.,0000001750,net income,NetIncomeLoss,Net Income (Loss) Attributable to Parent,USD,2025-06-01,2026-05-31,1.877000e+08,DEF 14A,NaN,None,2026-08-04,CY2025,0001140361-26-031078
3,AAR CORP.,0000001750,assets,Assets,Assets,USD,None,2026-02-28,3.332500e+09,10-Q,2026.0,Q3,2026-03-25,CY2026Q1I,0001104659-26-033973
4,AAR CORP.,0000001750,assets,Assets,Assets,USD,None,2026-05-31,3.355900e+09,10-K,2026.0,FY,2026-07-22,CY2026Q2I,0001104659-26-085459
5,AAR CORP.,0000001750,liabilities,Liabilities,Liabilities,USD,None,2023-02-28,1.800000e+06,10-K,2023.0,FY,2023-07-18,None,0001104659-23-082069
6,AAR CORP.,0000001750,liabilities,Liabilities,Liabilities,USD,None,2023-02-28,1.800000e+06,10-K,2024.0,FY,2024-07-19,CY2023Q1I,0001104659-24-080890
7,AAR CORP.,0000001750,equity,StockholdersEquity,Stockholders' Equity Attributable to Parent,USD,None,2026-02-28,1.643400e+09,10-Q,2026.0,Q3,2026-03-25,CY2026Q1I,0001104659-26-033973
8,AAR CORP.,0000001750,equity,StockholdersEquity,Stockholders' Equity Attributable to Parent,USD,None,2026-05-31,1.703800e+09,10-K,2026.0,FY,2026-07-22,CY2026Q2I,0001104659-26-085459
9,AAR CORP.,0000001750,cash,CashAndCashEquivalentsAtCarryingValue,"Cash and Cash Equivalents, at Carrying Value",USD,None,2026-02-28,7.850000e+07,10-Q,2026.0,Q3,2026-03-25,CY2026Q1I,0001104659-26-033973


#### ABBOTT LABORATORIES

,company,cik,metric,concept,label,unit,start,end,value,form,fiscal_year,fiscal_period,filed,frame,accession_number
0,ABBOTT LABORATORIES,0000001800,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,"Revenue from Contract with Customer, Excluding Assessed Tax",USD,2025-01-01,2025-12-31,4.432800e+10,10-K,2025.0,FY,2026-02-20,CY2025,0001628280-26-010185
1,ABBOTT LABORATORIES,0000001800,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,"Revenue from Contract with Customer, Excluding Assessed Tax",USD,2026-01-01,2026-03-31,1.116400e+10,10-Q,2026.0,Q1,2026-04-29,CY2026Q1,0001628280-26-028357
2,ABBOTT LABORATORIES,0000001800,net income,NetIncomeLoss,Net Income (Loss) Attributable to Parent,USD,2025-01-01,2025-12-31,6.524000e+09,10-K,2025.0,FY,2026-02-20,CY2025,0001628280-26-010185
3,ABBOTT LABORATORIES,0000001800,net income,NetIncomeLoss,Net Income (Loss) Attributable to Parent,USD,2026-01-01,2026-03-31,1.077000e+09,10-Q,2026.0,Q1,2026-04-29,CY2026Q1,0001628280-26-028357
4,ABBOTT LABORATORIES,0000001800,assets,Assets,Assets,USD,None,2025-12-31,8.671300e+10,10-Q,2026.0,Q1,2026-04-29,CY2025Q4I,0001628280-26-028357
5,ABBOTT LABORATORIES,0000001800,assets,Assets,Assets,USD,None,2026-03-31,1.104290e+11,10-Q,2026.0,Q1,2026-04-29,CY2026Q1I,0001628280-26-028357
6,ABBOTT LABORATORIES,0000001800,equity,StockholdersEquity,Stockholders' Equity Attributable to Parent,USD,None,2025-12-31,5.213000e+10,10-Q,2026.0,Q1,2026-04-29,CY2025Q4I,0001628280-26-028357
7,ABBOTT LABORATORIES,0000001800,equity,StockholdersEquity,Stockholders' Equity Attributable to Parent,USD,None,2026-03-31,5.206100e+10,10-Q,2026.0,Q1,2026-04-29,CY2026Q1I,0001628280-26-028357
8,ABBOTT LABORATORIES,0000001800,cash,CashAndCashEquivalentsAtCarryingValue,"Cash and Cash Equivalents, at Carrying Value",USD,None,2025-12-31,8.522000e+09,10-Q,2026.0,Q1,2026-04-29,CY2025Q4I,0001628280-26-028357
9,ABBOTT LABORATORIES,0000001800,cash,CashAndCashEquivalentsAtCarryingValue,"Cash and Cash Equivalents, at Carrying Value",USD,None,2026-03-31,6.803000e+09,10-Q,2026.0,Q1,2026-04-29,CY2026Q1I,0001628280-26-028357


### 7. Временное покрытие

`median_step_days` и `most_common_step_days` рассчитаны по отсортированным уникальным `end`. Периодичность сначала определяется для каждого `concept + unit`; сводная периодичность компании становится `mixed`, если существенные ряды неоднородны.

In [7]:
def cadence_from_dates(date_series):
    dates = pd.Series(pd.to_datetime(date_series, errors="coerce")).dropna().drop_duplicates().sort_values()
    if len(dates) < 2:
        return "irregular", np.nan, np.nan
    diffs = dates.diff().dt.days.dropna()
    median_step = float(diffs.median())
    modes = diffs.mode()
    common_step = float(modes.iloc[0]) if not modes.empty else np.nan
    quarterly_share = diffs.between(80, 100).mean()
    annual_share = diffs.between(340, 390).mean()
    if quarterly_share >= 0.5 or 80 <= median_step <= 100:
        cadence = "quarterly"
    elif annual_share >= 0.5 or 340 <= median_step <= 390:
        cadence = "annual"
    elif quarterly_share > 0 and annual_share > 0:
        cadence = "mixed"
    else:
        cadence = "irregular"
    return cadence, median_step, common_step

coverage_cu_rows = []
valid_end_facts = matched_facts[matched_facts["end_date"].notna()].copy()
for keys, group in valid_end_facts.groupby(["cik", "entity_name", "taxonomy", "concept", "unit"], dropna=False):
    cadence, median_step, common_step = cadence_from_dates(group["end_date"])
    dates = group["end_date"].drop_duplicates().sort_values()
    coverage_cu_rows.append({
        "cik": keys[0], "company": keys[1], "taxonomy": keys[2], "concept": keys[3], "unit": keys[4],
        "earliest_end_date": dates.min(), "latest_end_date": dates.max(), "unique_end_dates": len(dates),
        "median_step_days": median_step, "most_common_step_days": common_step, "cadence": cadence,
        "max_gap_days": dates.diff().dt.days.max() if len(dates) > 1 else np.nan,
    })
coverage_cu_columns = ["cik", "company", "taxonomy", "concept", "unit", "earliest_end_date", "latest_end_date", "unique_end_dates", "median_step_days", "most_common_step_days", "cadence", "max_gap_days"]
coverage_by_concept_unit = pd.DataFrame(coverage_cu_rows, columns=coverage_cu_columns)

def company_cadence(cik):
    cadences = coverage_by_concept_unit.loc[
        (coverage_by_concept_unit["cik"] == cik) & (coverage_by_concept_unit["unique_end_dates"] >= 3), "cadence"
    ]
    if cadences.empty:
        return "irregular"
    shares = cadences.value_counts(normalize=True)
    if {"quarterly", "annual"}.issubset(set(cadences)) or shares.iloc[0] < 0.70:
        return "mixed"
    return shares.index[0]

coverage_rows = []
for (cik, company), group in valid_end_facts.groupby(["cik", "entity_name"], dropna=False):
    dates = group["end_date"].drop_duplicates().sort_values()
    _, median_step, common_step = cadence_from_dates(dates)
    coverage_rows.append({
        "company": company,
        "cik": cik,
        "earliest_end_date": dates.min(),
        "latest_end_date": dates.max(),
        "history_years": (dates.max() - dates.min()).days / 365.25 if len(dates) else np.nan,
        "unique_end_dates": len(dates),
        "median_step_days": median_step,
        "most_common_step_days": common_step,
        "cadence": company_cadence(cik),
        "earliest_filed_date": group["filed_date"].min(),
        "latest_filed_date": group["filed_date"].max(),
        "concept_count": group[["taxonomy", "concept"]].drop_duplicates().shape[0],
        "fact_count": len(group),
        "max_concept_gap_days": coverage_by_concept_unit.loc[coverage_by_concept_unit["cik"] == cik, "max_gap_days"].max(),
    })
coverage_columns = ["company", "cik", "earliest_end_date", "latest_end_date", "history_years", "unique_end_dates", "median_step_days", "most_common_step_days", "cadence", "earliest_filed_date", "latest_filed_date", "concept_count", "fact_count", "max_concept_gap_days"]
company_coverage = pd.DataFrame(coverage_rows, columns=coverage_columns).sort_values("history_years", ascending=False)
display(company_coverage.drop(columns="max_concept_gap_days").style.format({
    "history_years": "{:.1f}", "median_step_days": "{:.1f}", "most_common_step_days": "{:.0f}",
}))

if not company_coverage.empty:
    overall_coverage = pd.DataFrame({
        "metric": ["Самая ранняя end", "Самая поздняя end", "Медианная история, лет", "Минимальная история, лет", "Максимальная история, лет"],
        "value": [company_coverage["earliest_end_date"].min(), company_coverage["latest_end_date"].max(), company_coverage["history_years"].median(), company_coverage["history_years"].min(), company_coverage["history_years"].max()],
    })
    display(overall_coverage)
    history_thresholds = pd.DataFrame({
        "minimum_years": [5, 10, 15, 20],
        "company_count": [(company_coverage["history_years"] >= y).sum() for y in [5, 10, 15, 20]],
        "share": [(company_coverage["history_years"] >= y).mean() for y in [5, 10, 15, 20]],
    })
    display(history_thresholds.style.format({"share": "{:.1%}"}))
    print("Самая длинная история:")
    display(company_coverage.nlargest(5, "history_years")[["company", "cik", "earliest_end_date", "latest_end_date", "history_years"]])
    print("Самая короткая история:")
    display(company_coverage.nsmallest(5, "history_years")[["company", "cik", "earliest_end_date", "latest_end_date", "history_years"]])
    print("Наибольшие разрывы внутри concept+unit:")
    display(company_coverage.nlargest(10, "max_concept_gap_days")[["company", "cik", "max_concept_gap_days"]])

,company,cik,earliest_end_date,latest_end_date,history_years,unique_end_dates,median_step_days,most_common_step_days,cadence,earliest_filed_date,latest_filed_date,concept_count,fact_count


### 8. Распределения временного покрытия

Графики относятся только к подтверждённо сопоставленным компаниям. Ось количества означает число компаний.

In [8]:
if not company_coverage.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    start_year_counts = company_coverage["earliest_end_date"].dt.year.value_counts().sort_index()
    axes[0].bar(start_year_counts.index.astype(str), start_year_counts.values, color="#4C78A8")
    axes[0].set_title("Компании по году начала наблюдений")
    axes[0].set_xlabel("Год первой end-даты")
    axes[0].set_ylabel("Количество компаний")
    axes[0].tick_params(axis="x", rotation=70)

    cadence_counts = company_coverage["cadence"].value_counts()
    axes[1].bar(cadence_counts.index, cadence_counts.values, color="#F58518")
    axes[1].set_title("Типичная периодичность компаний")
    axes[1].set_xlabel("Cadence")
    axes[1].set_ylabel("Количество компаний")
    plt.tight_layout()
    plt.show()
else:
    print("Нет подтверждённых компаний для построения графиков.")

Нет подтверждённых компаний для построения графиков.


### 9. Качество данных

Полный уровень файлов: читаемость, отсутствие CIK/названия и дубликаты CIK. Уровень фактов: подтверждённые компании из `train.csv`; если их невозможно подтвердить без справочника, детальные проверки явно ограничены двумя иллюстративными JSON-компаниями. Композитный ключ дубля включает как минимум `cik + taxonomy + concept + unit + start + end + form + filed + accn`. Записи не удаляются.

In [9]:
facts = quality_facts.copy()
print("Fact-level scope:", quality_scope_label, f"({facts['cik'].nunique()} CIK, {len(facts):,} facts)")
fact_key = ["cik", "taxonomy", "concept", "unit", "start", "end", "form", "filed", "accn"]
for col in fact_key:
    facts[col] = facts[col].astype("string")

exact_key_duplicate_mask = facts.duplicated(fact_key, keep=False)
duplicate_fact_rows = facts[exact_key_duplicate_mask].sort_values(fact_key)
duplicate_key_groups = facts.groupby(fact_key, dropna=False).size()
duplicate_key_groups = duplicate_key_groups[duplicate_key_groups > 1]

period_key = ["cik", "taxonomy", "concept", "unit", "start", "end", "form"]
publication_counts = facts.groupby(period_key, dropna=False).agg(
    rows=("accn", "size"), distinct_accessions=("accn", "nunique"), distinct_filed=("filed", "nunique")
).reset_index()
repeated_publications = publication_counts[(publication_counts["distinct_accessions"] > 1) | (publication_counts["distinct_filed"] > 1)]

value_check = facts.assign(value_text=facts["val"].astype(str)).groupby(
    ["cik", "taxonomy", "concept", "unit", "end"], dropna=False
).agg(rows=("value_text", "size"), distinct_values=("value_text", "nunique"), distinct_accessions=("accn", "nunique"), earliest_filed=("filed_date", "min"), latest_filed=("filed_date", "max")).reset_index()
multiple_values_same_end = value_check[(value_check["distinct_values"] > 1) & (value_check["distinct_accessions"] > 1)]

unit_check = facts.groupby(["cik", "taxonomy", "concept"], dropna=False)["unit"].nunique().reset_index(name="unit_count")
multi_unit_concepts = unit_check[unit_check["unit_count"] > 1]

numeric_values = pd.to_numeric(facts["val"], errors="coerce")
duration_days = (facts["end_date"] - facts["start_date"]).dt.days
six_month_cumulative = facts[duration_days.between(150, 210, inclusive="both")]
nine_month_cumulative = facts[duration_days.between(240, 300, inclusive="both")]
filing_lag_days = (facts["filed_date"] - facts["end_date"]).dt.days

potential_restatements = multiple_values_same_end[multiple_values_same_end["latest_filed"] > multiple_values_same_end["earliest_filed"]].copy()

quality_summary = pd.DataFrame({
    "scope": ["all JSON", "all JSON", "all JSON"] + [quality_scope_label] * 8,
    "check": [
        "Unreadable/corrupt JSON files", "Files missing CIK", "Files missing entityName",
        "Facts with missing/invalid end", "Facts with missing/invalid filed", "Rows in duplicated composite keys",
        "Duplicated composite-key groups", "Repeated publication period groups", "concept+unit+end groups with multiple values/reports",
        "Concepts reported in multiple units", "Negative numeric facts (not automatically errors)",
    ],
    "count": [
        len(read_errors), catalog["cik"].isna().sum(), catalog["entityName"].fillna("").astype(str).str.strip().eq("").sum(),
        facts["end_date"].isna().sum(), facts["filed_date"].isna().sum(), exact_key_duplicate_mask.sum(),
        len(duplicate_key_groups), len(repeated_publications), len(multiple_values_same_end),
        len(multi_unit_concepts), (numeric_values < 0).sum(),
    ],
    "denominator": [
        len(json_files), len(catalog), len(catalog), len(facts), len(facts), len(facts),
        len(facts), len(publication_counts), len(value_check), len(unit_check), numeric_values.notna().sum(),
    ],
})
quality_summary["rate"] = quality_summary["count"] / quality_summary["denominator"].replace(0, np.nan)
display(quality_summary.style.format({"count": "{:,}", "denominator": "{:,}", "rate": "{:.2%}"}))

temporal_quality = pd.DataFrame({
    "metric": ["6-month cumulative observations", "9-month cumulative observations", "Median filed-minus-end lag, days", "95th percentile filing lag, days", "Potential restatement groups"],
    "value": [len(six_month_cumulative), len(nine_month_cumulative), filing_lag_days.median(), filing_lag_days.quantile(0.95), len(potential_restatements)],
})
display(temporal_quality)

Fact-level scope: 2 illustrative JSON companies (no confirmed train matches) (2 CIK, 43,546 facts)


,scope,check,count,denominator,rate
0,all JSON,Unreadable/corrupt JSON files,0,"20,290",0.00%
1,all JSON,Files missing CIK,70,"20,290",0.34%
2,all JSON,Files missing entityName,"2,490","20,290",12.27%
3,2 illustrative JSON companies (no confirmed train matches),Facts with missing/invalid end,0,"43,546",0.00%
4,2 illustrative JSON companies (no confirmed train matches),Facts with missing/invalid filed,0,"43,546",0.00%
5,2 illustrative JSON companies (no confirmed train matches),Rows in duplicated composite keys,0,"43,546",0.00%
6,2 illustrative JSON companies (no confirmed train matches),Duplicated composite-key groups,0,"43,546",0.00%
7,2 illustrative JSON companies (no confirmed train matches),Repeated publication period groups,"14,123","24,990",56.51%
8,2 illustrative JSON companies (no confirmed train matches),concept+unit+end groups with multiple values/reports,"3,185","20,201",15.77%
9,2 illustrative JSON companies (no confirmed train matches),Concepts reported in multiple units,7,979,0.72%


,metric,value
0,6-month cumulative observations,3930.0
1,9-month cumulative observations,4010.0
2,"Median filed-minus-end lag, days",126.0
3,"95th percentile filing lag, days",779.0
4,Potential restatement groups,3185.0


### 10. Примеры проблемных grain и повторных публикаций

Несколько значений для одного `concept + unit + end` не являются автоматически ошибкой: причины включают разные длительности периода, формы, dimensions, amendments и restatements. Для point-in-time анализа необходимо выбирать только записи, опубликованные (`filed`) к соответствующей модельной дате, и сохранять accession/form/context.

In [10]:
print("Примеры дублированных композитных ключей:")
display(duplicate_fact_rows[fact_key + ["val", "frame"]].head(20))
print("Примеры нескольких значений одного concept+unit+end из разных отчётов:")
display(multiple_values_same_end.sort_values(["distinct_values", "rows"], ascending=False).head(20))
print("Примеры потенциальных restatements:")
display(potential_restatements.sort_values("latest_filed", ascending=False).head(20))
print("Примеры концептов с несколькими единицами:")
display(multi_unit_concepts.sort_values("unit_count", ascending=False).head(20))

Примеры дублированных композитных ключей:


,cik,taxonomy,concept,unit,start,end,form,filed,accn,val,frame


Примеры нескольких значений одного concept+unit+end из разных отчётов:


,cik,taxonomy,concept,unit,end,rows,distinct_values,distinct_accessions,earliest_filed,latest_filed
19117,0000001800,us-gaap,SalesRevenueNet,USD,2012-09-30,6,6,4,2012-11-07,2014-02-21
19121,0000001800,us-gaap,SalesRevenueNet,USD,2013-09-30,6,6,4,2013-11-07,2015-02-27
3550,0000001750,us-gaap,GrossProfit,USD,2011-11-30,6,5,4,2011-12-22,2013-07-26
3627,0000001750,us-gaap,IncomeLossFromContinuingOperations,USD,2017-11-30,6,5,4,2017-12-22,2019-07-18
8666,0000001750,us-gaap,SalesRevenueNet,USD,2011-11-30,6,5,4,2011-12-22,2013-07-26
14104,0000001800,us-gaap,IncomeLossFromContinuingOperationsIncludingPortionAttributableToNoncontrollingInterest,USD,2013-06-30,6,5,4,2013-08-06,2015-02-27
14105,0000001800,us-gaap,IncomeLossFromContinuingOperationsIncludingPortionAttributableToNoncontrollingInterest,USD,2013-09-30,6,5,4,2013-11-07,2015-02-27
19116,0000001800,us-gaap,SalesRevenueNet,USD,2012-06-30,6,5,4,2012-08-07,2014-02-21
19120,0000001800,us-gaap,SalesRevenueNet,USD,2013-06-30,6,5,4,2013-08-06,2015-02-27
3568,0000001750,us-gaap,GrossProfit,USD,2016-05-31,5,5,3,2016-07-13,2018-07-11


Примеры потенциальных restatements:


,cik,taxonomy,concept,unit,end,rows,distinct_values,distinct_accessions,earliest_filed,latest_filed
3302,0000001750,us-gaap,FiniteLivedIntangibleAssetsAccumulatedAmortization,USD,2025-05-31,3,2,3,2025-07-22,2026-07-22
2369,0000001750,us-gaap,DeferredTaxAssetsGross,USD,2025-05-31,2,2,2,2025-07-22,2026-07-22
4587,0000001750,us-gaap,IncreaseDecreaseInOtherOperatingCapitalNet,USD,2025-05-31,2,2,2,2025-07-22,2026-07-22
4611,0000001750,us-gaap,IncreaseDecreaseInPrepaidExpensesOther,USD,2024-05-31,3,2,3,2024-07-19,2026-07-22
7696,0000001750,us-gaap,PrepaidExpenseAndOtherAssetsCurrent,USD,2025-05-31,5,2,5,2025-07-22,2026-07-22
4615,0000001750,us-gaap,IncreaseDecreaseInPrepaidExpensesOther,USD,2025-05-31,2,2,2,2025-07-22,2026-07-22
3188,0000001750,us-gaap,EffectiveIncomeTaxRateReconciliationOtherAdjustments,pure,2024-05-31,3,2,3,2024-07-19,2026-07-22
2671,0000001750,us-gaap,DepreciationDepletionAndAmortization,USD,2024-05-31,3,2,3,2024-07-19,2026-07-22
2608,0000001750,us-gaap,DefinedContributionPlanEmployerDiscretionaryContributionAmount,USD,2024-05-31,3,2,3,2024-07-19,2026-07-22
4583,0000001750,us-gaap,IncreaseDecreaseInOtherOperatingCapitalNet,USD,2024-05-31,3,3,3,2024-07-19,2026-07-22


Примеры концептов с несколькими единицами:


,cik,taxonomy,concept,unit_count
135,0000001750,us-gaap,DerivativeNumberOfInstrumentsHeld,3
334,0000001750,us-gaap,NumberOfReportableSegments,2
335,0000001750,us-gaap,NumberOfReportingUnits,2
677,0000001800,us-gaap,FiniteLivedIntangibleAssetsAverageUsefulLife,2
682,0000001800,us-gaap,FiniteLivedIntangibleAssetsUsefulLifeMaximum,2
683,0000001800,us-gaap,FiniteLivedIntangibleAssetsUsefulLifeMinimum,2
937,0000001800,us-gaap,ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsExercisableWeightedAve...,2


## Takeaways

### 11. Итог и пригодность для моделирования

Итог формируется непосредственно из рассчитанных выше таблиц, поэтому цифры синхронизированы с выполненным notebook.

In [11]:
if not company_coverage.empty:
    earliest = company_coverage["earliest_end_date"].min().date()
    latest = company_coverage["latest_end_date"].max().date()
    dominant_cadence = company_coverage["cadence"].value_counts().idxmax()
    median_years = company_coverage["history_years"].median()
    median_years_text = f"{median_years:.1f} лет"
else:
    earliest = latest = "н/д"
    dominant_cadence = "н/д"
    median_years = np.nan
    median_years_text = "н/д"

conclusion = f'''
### Краткий вывод

- **Покрытие:** подтверждено **{match_count} из {train_company_count}** компаний/идентификаторов train (**{match_rate:.1%}**); неоднозначные ticker–CIK не включались.
- **Доступные даты:** среди подтверждённых компаний `end` охватывает **{earliest} — {latest}**, медианная продолжительность истории — **{median_years_text}**.
- **Периодичность:** наиболее частая сводная категория — **{dominant_cadence}**. Классификация учитывает различия между `concept + unit`, поэтому смешанные компании не маскируются общим набором дат.
- **Основные риски до модели:** повторные публикации и потенциальные restatements, смешанный grain (instant/duration, формы и frames), разные единицы, накопительные 6/9-месячные значения и look-ahead через дату `filed`.
- **Panel dataset:** единый фундаментальный panel нельзя надёжно получить простой склейкой. Нужны point-in-time правила (`filed`/`accn`), нормализация синонимов XBRL-концептов, единиц, длительности периода и dimensions; универсальный набор метрик будет иметь неполное покрытие.
- **Качество источника:** нечитаемых JSON — **{len(read_errors)}**; fact-level scope — **{quality_scope_label}**, **{facts['cik'].nunique()} CIK / {len(facts):,} фактов**.
'''
display(Markdown(conclusion))


### Краткий вывод

- **Покрытие:** подтверждено **0 из 50** компаний/идентификаторов train (**0.0%**); неоднозначные ticker–CIK не включались.
- **Доступные даты:** среди подтверждённых компаний `end` охватывает **н/д — н/д**, медианная продолжительность истории — **н/д**.
- **Периодичность:** наиболее частая сводная категория — **н/д**. Классификация учитывает различия между `concept + unit`, поэтому смешанные компании не маскируются общим набором дат.
- **Основные риски до модели:** повторные публикации и потенциальные restatements, смешанный grain (instant/duration, формы и frames), разные единицы, накопительные 6/9-месячные значения и look-ahead через дату `filed`.
- **Panel dataset:** единый фундаментальный panel нельзя надёжно получить простой склейкой. Нужны point-in-time правила (`filed`/`accn`), нормализация синонимов XBRL-концептов, единиц, длительности периода и dimensions; универсальный набор метрик будет иметь неполное покрытие.
- **Качество источника:** нечитаемых JSON — **0**; fact-level scope — **2 illustrative JSON companies (no confirmed train matches)**, **2 CIK / 43,546 фактов**.


### Рекомендуемые правила перед обучением модели

1. Строить point-in-time срез только из фактов с `filed <= prediction_date`; хранить `accn`, `form`, `frame` и признак amendment.
2. Задать канонический словарь метрик (например, несколько revenue-концептов → одна бизнес-метрика) с явным приоритетом и контролем taxonomy.
3. Разделять instant и duration facts; для duration нормализовать квартал, 6M, 9M и FY, не смешивая накопительные значения.
4. Приводить единицы и масштабы, но не объединять разные units без бизнес-правила.
5. Автоматизировать проверки: JSON parse, not-null CIK/name, однозначность ticker–CIK, композитные дубли, multiple values per period и отрицательный filing lag.